# P2-A2 — Structured Output (turn an LLM into a programmable component)

In P2-A1 you wrote: *"the output isn't the final answer, it's an input to the next step."* This task makes that real. You'll get the model to return **structured JSON** you can `json.loads()` and load straight into pandas — the backbone of every LLM-powered feature (extraction, classification pipelines, agents reading tool results).

Goal: go from *"the model wrote a nice paragraph"* to *"the model returned data my program can use."*

In [ ]:
# --- Setup (given) ---
import json
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o-mini'

def ask(prompt, system=None, **kwargs):
    messages = []
    if system:
        messages.append({'role': 'system', 'content': system})
    messages.append({'role': 'user', 'content': prompt})
    resp = client.chat.completions.create(model=MODEL, messages=messages, **kwargs)
    return resp.choices[0].message.content

# --- Given: raw, messy customer reviews to extract data from ---
reviews = [
    "Absolutely love this blender! Arrived a day early and works like a charm. Worth every penny.",
    "The laptop stand is fine I guess, but it took three weeks to ship and the box was crushed.",
    "Terrible. The headphones stopped working after two days and support never replied. Avoid.",
    "Decent coffee mug. Nothing fancy, does the job, shipping was on time.",
]
print(len(reviews), 'reviews loaded')

## Task 1 — First attempt: ask for JSON, then parse it
Write a prompt that asks the model to extract, from **one** review, a JSON object with these fields:
`product` (string), `sentiment` (positive/negative/neutral), `shipping_mentioned` (true/false).
<br>Then run `json.loads(...)` on the reply and print the resulting dict.
<br>Goal: see that a plain "return JSON" prompt *often* works — but is fragile (stray text, markdown fences, etc. can break `json.loads`).

In [ ]:
# TODO: prompt for JSON on reviews[0], then json.loads() it and print the dict


## Task 2 — Make it reliable: JSON mode + a clear schema
OpenAI can *guarantee* valid JSON if you pass `response_format={'type': 'json_object'}` to the call (your `ask()` helper forwards `**kwargs`, so you can pass it straight through). Combine that with a prompt that spells out the exact field names.
<br>Re-extract `reviews[0]` this way and `json.loads` it — it should parse every time.
<br>Hint: when using JSON mode, your prompt must mention the word "JSON" and describe the fields you want.

In [ ]:
# TODO: same extraction but with response_format={'type': 'json_object'}


## Task 3 — Process all reviews into a DataFrame
Run your reliable extraction over **all** `reviews`, collect the dicts into a list, and build a pandas `DataFrame` from them.
<br>(A `for` loop is fine here — each call is a network request, not a vectorizable array op.)
<br>Goal: this is a real mini-pipeline — unstructured text in, clean table out. Exactly what you'd build at work.

In [ ]:
# TODO: extract all reviews -> list of dicts -> pd.DataFrame


## Task 4 — Explain (in your own words)
1. What did `response_format={'type': 'json_object'}` change vs. just asking for JSON in Task 1?
2. Even with guaranteed-valid JSON, what can still go *wrong* with the data (think: the model is guessing the values)? Name one thing you'd add to a production pipeline to guard against it.

> _your answer here_